# Is the DP failure the privacy budget, or the configuration?

Section 5.11 reports both aggregators falling to chance under DP-SGD at
every budget tested, and concludes only that AHA-Fed is architecturally
compatible with per-client DP. That conclusion rests on runs configured
with 300 images per client over 15 rounds, against 1,000 images and 25
rounds for every non-private result in the paper.

That is not a like-for-like comparison, and Section 5.9 has since shown
per-client sample size is the variable that governs whether the quality
signal works at all. DP-SGD is also the setting most sensitive to it:
the noise added per step is calibrated to the sampling rate, so a client
holding 300 images and stepping on batches of 64 gets a poor
signal-to-noise ratio before privacy is even accounted for.

This repeats the sweep at the paper's standard budget -- 1,000 images
per client, 25 rounds -- and adds a larger batch, which is standard
DP-SGD practice because it improves the privacy/utility trade at fixed
epsilon. Three seeds per configuration.

If utility stays at chance, the negative result is about the privacy
budget and the paper's conclusion is unchanged but now properly
supported. If utility recovers, the earlier finding was a configuration
artifact and Section 5.11 has to be rewritten.


In [ ]:
import glob, os, shutil, sys
REPO='/kaggle/working/repo'
hits=glob.glob('/kaggle/input/**/paper/build_paper.py', recursive=True)
assert hits, 'attach the aha-fed-code dataset'
src=os.path.dirname(os.path.dirname(hits[0]))
if not os.path.isdir(os.path.join(REPO,'paper')):
    shutil.copytree(src, REPO, dirs_exist_ok=True)
os.chdir(REPO); sys.path.insert(0, REPO)
print('staged:', os.getcwd())


In [ ]:
!pip install -q medmnist 'opacus==1.5.4'

In [ ]:
import subprocess, sys
PROBE=('import torch;'
       'a=torch.randn(4,3,8,8,device="cuda");'
       'w=torch.randn(2,3,3,3,device="cuda");'
       'torch.nn.functional.conv2d(a,w).sum().item();'
       'print(torch.cuda.get_device_name(0), torch.__version__)')
r=subprocess.run([sys.executable,'-c',PROBE],capture_output=True,text=True)
print(r.stdout.strip() or r.stderr.strip()[-400:])
if r.returncode!=0:
    subprocess.run([sys.executable,'-m','pip','install','-q',
                    'torch==2.4.1','torchvision==0.19.1',
                    '--index-url','https://download.pytorch.org/whl/cu118'],check=True)
    # opacus binds to the torch it was built against, so reinstall the
    # pinned version after any swap.
    subprocess.run([sys.executable,'-m','pip','install','-q',
                    '--force-reinstall','--no-deps','opacus==1.5.4'],check=True)
    r=subprocess.run([sys.executable,'-c',PROBE],capture_output=True,text=True)
    print(r.stdout.strip() or r.stderr.strip()[-400:])
    if r.returncode!=0: raise SystemExit('no working torch/GPU combination')

# Fail fast if the DP stack is broken, and say which versions are in play:
# version 1 died here on opacus importing DTensor from a torch that has none.
CHECK=('import torch, opacus;'
       'print("opacus", opacus.__version__, "torch", torch.__version__);'
       'from opacus import PrivacyEngine;'
       'print("PrivacyEngine OK")')
r=subprocess.run([sys.executable,'-c',CHECK],capture_output=True,text=True)
print(r.stdout.strip() or r.stderr.strip()[-800:])
if r.returncode!=0: raise SystemExit('opacus import failed')


In [ ]:
import traceback
from paper.run_real_experiments import run_one, OUT_DIR

SEEDS=[42,123,456]
EPS=[1.0,10.0]   # opposite ends of the budget range answer the question

# The paper's standard budget: 1000 img/client, 25 rounds, against the
# 300/15 the original DP sweep used. Batch 128 is deliberate -- DP-SGD
# trades batch size against noise at fixed epsilon.
done=fail=0
for eps in EPS:
    dp=dict(epsilon=eps, delta=1e-5, max_grad_norm=1.0)
    for s in SEEDS:
        common=dict(dataset='pathmnist', num_classes=9,
                    dp_config=dp, num_rounds=25, samples_per_client=1000,
                    batch_size=128, local_lr=0.05, seed=s)
        for name, kw in ((f'dp2_fedavg_eps{eps}_s{s}', dict(aggregator='fedavg')),
                         (f'dp2_aha_eps{eps}_s{s}',
                          dict(aggregator='adaptive', alpha=1.0,
                               beta=4.0, gamma=4.0))):
            try:
                run_one(OUT_DIR/f'{name}.json', **kw, **common)
                done+=1
            except Exception:
                fail+=1
                print(f'FAILED {name}'); traceback.print_exc()
    print(f'eps={eps} done ({done} ok, {fail} failed)', flush=True)
print(f'TOTAL {done} ok, {fail} failed')


In [ ]:
import glob, json, re, statistics as st
from pathlib import Path

def by_seed(pat, key):
    out={}
    for f in sorted(glob.glob(pat)):
        d=json.loads(Path(f).read_text())
        fin=d.get('final') or {}
        m=re.search(r'_s(\d+)$', Path(f).stem)
        if fin.get(key) is not None:
            out[m.group(1) if m else '42']=fin[key]
    return out

CHANCE=1/9
print(f'chance accuracy on 9-class PathMNIST = {CHANCE:.3f}')
print('non-private reference: AHA-Fed 0.702, FedAvg 0.679')
print()
print('  eps    AHA acc   FedAvg acc   AHA ece   FedAvg ece   above chance?')
for eps in [1.0,10.0]:
    aa=by_seed(f'runs/paper/real/dp2_aha_eps{eps}_s*.json','accuracy')
    va=by_seed(f'runs/paper/real/dp2_fedavg_eps{eps}_s*.json','accuracy')
    ae=by_seed(f'runs/paper/real/dp2_aha_eps{eps}_s*.json','ece')
    ve=by_seed(f'runs/paper/real/dp2_fedavg_eps{eps}_s*.json','ece')
    c=sorted(set(aa)&set(va))
    if not c: continue
    ma, mv = st.mean([aa[k] for k in c]), st.mean([va[k] for k in c])
    print(f'  {eps:4}   {ma:.3f}     {mv:.3f}        '
          f'{st.mean([ae[k] for k in c]):.4f}    {st.mean([ve[k] for k in c]):.4f}     '
          f'{"YES" if max(ma,mv)>CHANCE*1.5 else "no"}  (n={len(c)})')

print()
print('old sweep for comparison (300 img/client, 15 rounds):')
for eps in ['0.5','1.0','5.0']:
    for agg in ['aha','fedavg']:
        f=f'runs/paper/real/path_{agg}_dp_eps{eps}.json'
        if Path(f).exists():
            d=json.loads(Path(f).read_text())
            print(f'  eps={eps} {agg:6s} acc={d["final"]["accuracy"]:.3f}')


In [ ]:
!cd /kaggle/working/repo && zip -qr /kaggle/working/dp2_results.zip runs/paper/real
!ls -lh /kaggle/working/dp2_results.zip
